<a href="https://colab.research.google.com/github/Tehila-BD/cloud-computing/blob/main/Tutorial7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# user_service.py
class UserService:
    def __init__(self):
        self.users = {
            '1': {'id': '1', 'name': 'John Doe', 'email': 'john@example.com'},
            '2': {'id': '2', 'name': 'Jane Doe', 'email': 'jane@example.com'}
        }

    def get_user(self, user_id):
        return self.users.get(user_id, {})

In [2]:
# index_service.py
class IndexService:
    def __init__(self):
        self.documents = {}
        self.index = {}

    def add_document(self, doc_data):
        """Add a document to the index"""
        doc_id = str(len(self.documents) + 1)
        self.documents[doc_id] = {**doc_data, 'id': doc_id}

        # Create inverted index
        words = doc_data['content'].lower().split()
        for word in words:
            if word not in self.index:
                self.index[word] = set()
            self.index[word].add(doc_id)

        return self.documents[doc_id]

    def get_document(self, doc_id):
        """Retrieve a document by ID"""
        return self.documents.get(doc_id)

    def search_word(self, word):
        """Find documents containing a word"""
        word = word.lower()
        return list(self.index.get(word, set()))

In [10]:
class QueryService:
    def __init__(self, index_service):
        self.index_service = index_service
        self.queries = {}

    def create_query(self, query_data):
        """Create and execute a search query with AND/OR and Relevance Ranking"""
        try:
            query_id = str(len(self.queries) + 1)
            search_terms = query_data['terms']
            operator = query_data.get('operator', 'AND').upper()

            # # Finding relevant documents by operator
            results = set()
            if operator == 'OR':
                for term in search_terms:
                    doc_ids = self.index_service.search_word(term)
                    results = results.union(set(doc_ids))
            else:
                first = True
                for term in search_terms:
                    doc_ids = self.index_service.search_word(term)
                    if first:
                        results = set(doc_ids)
                        first = False
                    else:
                        results &= set(doc_ids)

            # # The new ranking mechanism (Ranking)
            ranked_results = []
            for doc_id in results:
                # Pull out the full document to examine its contents
                doc = self.index_service.get_document(doc_id)
                content_lower = doc['content'].lower()

                # Count how many times the query words appear in total within the text of the document
                score = sum(content_lower.count(term.lower()) for term in search_terms)
                ranked_results.append((doc_id, score))

            # Sort the list of documents from highest to lowest score
            ranked_results.sort(key=lambda x: x[1], reverse=True)

            # Extracting only document IDs after they have been sorted
            final_sorted_doc_ids = [doc_id for doc_id, score in ranked_results]

            # Saving the query record
            query = {
                'id': query_id,
                'terms': search_terms,
                'operator': operator,
                'results': final_sorted_doc_ids,# Sorted list
                'timestamp': query_data.get('timestamp', 'now')
            }
            self.queries[query_id] = query
            return query

        except Exception as e:
            return {'error': str(e)}

In [11]:
class ResultService:
    def __init__(self, index_service, query_service):
        self.index_service = index_service
        self.query_service = query_service
        self.results = {}

    def format_results(self, query_id):
        """Format search results for display, including relevance scores"""
        try:
            query = self.query_service.queries.get(query_id)
            if not query:
                return {'error': 'Query not found'}

            formatted_results = []
            for doc_id in query['results']:
                doc = self.index_service.get_document(doc_id)
                if doc:
                    # Recalculating the score
                    content_lower = doc['content'].lower()
                    score = sum(content_lower.count(term.lower()) for term in query['terms'])

                    formatted_results.append({
                        'doc_id': doc_id,
                        'title': doc['title'],
                        'snippet': doc['content'][:100] + '...',
                        'relevance_score': score # Adding the score to the output
                    })

            result_id = str(len(self.results) + 1)
            result = {
                'id': result_id,
                'query_id': query_id,
                'formatted_results': formatted_results,
                'count': len(formatted_results)
            }
            self.results[result_id] = result
            return result

        except Exception as e:
            return {'error': str(e)}

In [12]:
index_service = IndexService()
query_service = QueryService(index_service)
result_service = ResultService(index_service, query_service)


index_service.add_document({
    'title': 'High Relevance Cloud',
    'content': 'Cloud computing is amazing. Cloud architecture is the future. Learn cloud today!'
})
index_service.add_document({
    'title': 'Medium Relevance Python & Cloud',
    'content': 'Python can be used with cloud services easily.'
})
index_service.add_document({
    'title': 'Low Relevance Python',
    'content': 'This is just a basic text talking about Python programming.'
})

print("Testing the ranking mechanism using the OR operator")
query = query_service.create_query({'terms': ['cloud', 'python'], 'operator': 'OR'})
formatted = result_service.format_results(query['id'])

for doc in formatted['formatted_results']:
    print(f"ID: {doc['doc_id']} | Title: {doc['title']} | Score: {doc['relevance_score']}")

Testing the ranking mechanism using the OR operator
ID: 1 | Title: High Relevance Cloud | Score: 3
ID: 2 | Title: Medium Relevance Python & Cloud | Score: 2
ID: 3 | Title: Low Relevance Python | Score: 1
